# Deep Learning for Energy Load Prediction

**Objective:** Predict Heating Load (Y1) and Cooling Load (Y2) using deep learning models.

**Frameworks:** TensorFlow/Keras and PyTorch
**Models:**
1. Vanilla MLP (Keras)
2. Deep Regularized MLP (Keras)
3. Residual MLP (Keras)
4. Deep MLP (PyTorch)
5. TabNet (PyTorch)

## Phase 1: Setup, Imports & Data Preprocessing

In [ ]:
# --- 0. Setup & Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau as TorchReduceLROnPlateau

try:
    from pytorch_tabnet.tab_model import TabNetRegressor
except ImportError:
    print("pytorch-tabnet not found. Installing...")
    !pip install pytorch-tabnet
    from pytorch_tabnet.tab_model import TabNetRegressor

# For reproducibility
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

print("TensorFlow Version:", tf.__version__)
print("PyTorch Version:", torch.__version__)
print("Setup Complete")

In [ ]:
# --- 1. Data Loading & Preprocessing ---

# Load the dataset
# The file is in the root, but our notebook is in 'analysis/', so we go up one level.
try:
    df = pd.read_excel('../ENB2012_data 2.xlsx')
except FileNotFoundError:
    print("Error: 'ENB2012_data 2.xlsx' not found in the parent directory.")
    # As a fallback, you might want to add code here to download it or point to the correct path.
    df = pd.DataFrame() # Empty dataframe to avoid further errors

if not df.empty:
    # Rename columns for easier access
    df.columns = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'Y1', 'Y2']

    # --- Data Integrity Checks ---
    print(f"Dataset Shape: {df.shape}")
    # 1. Check for missing values
    assert df.isnull().sum().sum() == 0, "Missing values found!"
    print("✅ No missing values.")

    # 2. Check for duplicated rows
    assert df.duplicated().sum() == 0, "Duplicate rows found!"
    print("✅ No duplicate rows.")

    # --- Feature and Target Split ---
    X = df[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']]
    y = df[['Y1', 'Y2']]

    # --- Train-Validation-Test Split (70/15/15) ---
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=(0.15/0.85), random_state=42) # 0.15/0.85 ensures val is 15% of total

    print(f"Train set size: {len(X_train)} ({len(X_train)/len(df):.0%})")
    print(f"Validation set size: {len(X_val)} ({len(X_val)/len(df):.0%})")
    print(f"Test set size: {len(X_test)} ({len(X_test)/len(df):.0%})")

    # --- Scaling --- 
    # Scale features
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_val_scaled = scaler_X.transform(X_val)
    X_test_scaled = scaler_X.transform(X_test)

    # Scale targets
    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train)
    y_val_scaled = scaler_y.transform(y_val)
    y_test_scaled = scaler_y.transform(y_test)
    
    print("\nData preprocessing complete.")

## Phase 2: Evaluation Utilities

In [ ]:
# --- 2. Evaluation Utilities ---

def evaluate_model(y_true, y_pred, model_name):
    """Calculates and prints regression metrics for a model's predictions."""
    # Inverse transform predictions and true values to original scale
    y_true_inv = scaler_y.inverse_transform(y_true)
    y_pred_inv = scaler_y.inverse_transform(y_pred)
    
    # Separate Y1 and Y2 for metric calculation
    y1_true, y2_true = y_true_inv[:, 0], y_true_inv[:, 1]
    y1_pred, y2_pred = y_pred_inv[:, 0], y_pred_inv[:, 1]
    
    metrics = {
        'RMSE_Y1': np.sqrt(mean_squared_error(y1_true, y1_pred)),
        'MAE_Y1': mean_absolute_error(y1_true, y1_pred),
        'R2_Y1': r2_score(y1_true, y1_pred),
        'RMSE_Y2': np.sqrt(mean_squared_error(y2_true, y2_pred)),
        'MAE_Y2': mean_absolute_error(y2_true, y2_pred),
        'R2_Y2': r2_score(y2_true, y2_pred),
    }
    
    print(f"--- Evaluation Metrics for {model_name} ---")
    print(f"Heating Load (Y1) - RMSE: {metrics['RMSE_Y1']:.4f}, MAE: {metrics['MAE_Y1']:.4f}, R²: {metrics['R2_Y1']:.4f}")
    print(f"Cooling Load (Y2) - RMSE: {metrics['RMSE_Y2']:.4f}, MAE: {metrics['MAE_Y2']:.4f}, R²: {metrics['R2_Y2']:.4f}")
    
    return metrics

def plot_history(history, model_name):
    """Plots training and validation loss from a Keras history object."""
    plt.figure(figsize=(12, 5))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Training and Validation Loss for {model_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_predictions(y_true, y_pred, model_name):
    """Plots predicted vs. actual values for Y1 and Y2."""
    y_true_inv = scaler_y.inverse_transform(y_true)
    y_pred_inv = scaler_y.inverse_transform(y_pred)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Y1: Heating Load
    ax1.scatter(y_true_inv[:, 0], y_pred_inv[:, 0], alpha=0.5, label='Predictions')
    ax1.plot([y_true_inv[:, 0].min(), y_true_inv[:, 0].max()], [y_true_inv[:, 0].min(), y_true_inv[:, 0].max()], 'r--', label='Ideal')
    ax1.set_title(f'{model_name} - Heating Load (Y1)')
    ax1.set_xlabel('Actual Values')
    ax1.set_ylabel('Predicted Values')
    ax1.legend()
    ax1.grid(True)
    
    # Y2: Cooling Load
    ax2.scatter(y_true_inv[:, 1], y_pred_inv[:, 1], alpha=0.5, label='Predictions')
    ax2.plot([y_true_inv[:, 1].min(), y_true_inv[:, 1].max()], [y_true_inv[:, 1].min(), y_true_inv[:, 1].max()], 'r--', label='Ideal')
    ax2.set_title(f'{model_name} - Cooling Load (Y2)')
    ax2.set_xlabel('Actual Values')
    ax2.set_ylabel('Predicted Values')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Store results for final comparison
results = {}

print("Evaluation utilities are ready.")